In [1]:
import os
import shutil
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
CSV_PATH = "metadata.csv"
IMG_DIR = "raw_data/train"

OUTPUT_DIR = "data"

TRAIN_DIR = os.path.join(OUTPUT_DIR, "train")
TEST_DIR = os.path.join(OUTPUT_DIR, "test")

TRAIN_IMG_DIR = os.path.join(TRAIN_DIR, "images")
TEST_IMG_DIR = os.path.join(TEST_DIR, "images")

os.makedirs(TRAIN_IMG_DIR, exist_ok=True)
os.makedirs(TEST_IMG_DIR, exist_ok=True)

In [3]:
df = pd.read_csv(CSV_PATH)
print(f"Original dataset size: {len(df)}")

Original dataset size: 33126


In [4]:
df.head()

,image_name,patient_id,lesion_id,sex,age_approx,anatom_site_general_challenge,diagnosis,benign_malignant,target
0,ISIC_2637011,IP_7279968,IL_7972535,male,45.0,head/neck,unknown,benign,0
1,ISIC_0015719,IP_3075186,IL_4649854,female,45.0,upper extremity,unknown,benign,0
2,ISIC_0052212,IP_2842074,IL_9087444,female,50.0,lower extremity,nevus,benign,0
3,ISIC_0068279,IP_6890425,IL_4255399,female,45.0,head/neck,unknown,benign,0
4,ISIC_0074268,IP_8723313,IL_6898037,female,55.0,upper extremity,unknown,benign,0


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33126 entries, 0 to 33125
Data columns (total 9 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   image_name                     33126 non-null  object 
 1   patient_id                     33126 non-null  object 
 2   lesion_id                      33126 non-null  object 
 3   sex                            33061 non-null  object 
 4   age_approx                     33058 non-null  float64
 5   anatom_site_general_challenge  32599 non-null  object 
 6   diagnosis                      33126 non-null  object 
 7   benign_malignant               33126 non-null  object 
 8   target                         33126 non-null  int64  
dtypes: float64(1), int64(1), object(7)
memory usage: 2.3+ MB


In [6]:
df.isnull().sum()

image_name                         0
patient_id                         0
lesion_id                          0
sex                               65
age_approx                        68
anatom_site_general_challenge    527
diagnosis                          0
benign_malignant                   0
target                             0
dtype: int64

In [7]:
df["sex"].unique()

array(['male', 'female', nan], dtype=object)

In [8]:
df["sex"] = df["sex"].fillna("unknown")

In [9]:
df["sex"].unique()

array(['male', 'female', 'unknown'], dtype=object)

In [10]:
df["age_approx"] = df["age_approx"].fillna(df["age_approx"].median())

In [11]:
df["anatom_site_general_challenge"] = (
    df["anatom_site_general_challenge"]
    .fillna("unknown")
)

df["anatom_site_general_challenge"].unique()

array(['head/neck', 'upper extremity', 'lower extremity', 'torso',
       'unknown', 'palms/soles', 'oral/genital'], dtype=object)

In [13]:
df["diagnosis"].unique()

array(['unknown', 'nevus', 'melanoma', 'seborrheic keratosis',
       'lentigo NOS', 'lichenoid keratosis', 'solar lentigo',
       'cafe-au-lait macule', 'atypical melanocytic proliferation'],
      dtype=object)

In [14]:
df["diagnosis"] = df["diagnosis"].fillna("unknown")

In [15]:
df.isnull().sum()

image_name                       0
patient_id                       0
lesion_id                        0
sex                              0
age_approx                       0
anatom_site_general_challenge    0
diagnosis                        0
benign_malignant                 0
target                           0
dtype: int64

In [16]:
len(df[df["benign_malignant"] == "benign"])

32542

In [17]:
len(df[df["benign_malignant"] == "malignant"])

584

In [18]:
len(df[df["target"] == 1])

584

In [19]:
len(df[df["target"] == 0])

32542

In [21]:
assert len(df[df["benign_malignant"] == "benign"]) == len(df[df["target"] == 0])

In [22]:
assert len(df[df["benign_malignant"] == "malignant"]) == len(df[df["target"] == 1])

In [23]:
malignant_df = df[df["target"] == 1]
benign_df = df[df["target"] == 0]

In [34]:
benign_sampled = benign_df.sample(frac=0.075, random_state=42)

In [35]:
filtered_df = pd.concat(
    [malignant_df, benign_sampled],
    ignore_index=True)

In [36]:
filtered_df = filtered_df.sample(
    frac=1, random_state=42
).reset_index(drop=True)

In [37]:
print(f"Filtered dataset size: {len(filtered_df)}")

Filtered dataset size: 3025


In [38]:
print("Target distribution:")
print(filtered_df["target"].value_counts())

Target distribution:
target
0    2441
1     584
Name: count, dtype: int64


In [39]:
train_df, test_df = train_test_split(
    filtered_df, test_size=0.25,
    stratify=filtered_df["target"], random_state=42)

In [40]:
print(f"Train size: {len(train_df)}")
print(f"Test size: {len(test_df)}")

Train size: 2268
Test size: 757


In [41]:
train_csv_path = os.path.join(TRAIN_DIR, "train_metadata.csv")
test_csv_path = os.path.join(TEST_DIR, "test_metadata.csv")

In [42]:
os.makedirs(TRAIN_DIR, exist_ok=True)
os.makedirs(TEST_DIR, exist_ok=True)

In [43]:
train_df.to_csv(train_csv_path, index=False)
test_df.to_csv(test_csv_path, index=False)

In [44]:
missing_train = 0

In [45]:
for img_name in train_df["image_name"]:
    src = os.path.join(IMG_DIR, img_name + ".jpg")
    dst = os.path.join(TRAIN_IMG_DIR, img_name + ".jpg")
    
    if os.path.exists(src):
        shutil.copy2(src, dst)
    else:
        missing_train += 1

In [46]:
missing_test = 0

In [47]:
for img_name in test_df["image_name"]:
    src = os.path.join(IMG_DIR, img_name + ".jpg")
    dst = os.path.join(TEST_IMG_DIR, img_name + ".jpg")

    if os.path.exists(src):
        shutil.copy2(src, dst)
    else:
        missing_test += 1

In [48]:
print(f"Missing train images: {missing_train}")
print(f"Missing test images: {missing_test}")

Missing train images: 0
Missing test images: 0


In [49]:
def get_folder_size(folder_path):
    total_size = 0

    for dirpath, dirnames, filenames in os.walk(folder_path):
        for f in filenames:
            fp = os.path.join(dirpath, f)

            if os.path.exists(fp):
                total_size += os.path.getsize(fp)

    return total_size

In [50]:
folder = "data"

size_bytes = get_folder_size(folder)

size_mb = size_bytes / (1024 ** 2)
size_gb = size_bytes / (1024 ** 3)

print(f"Folder: {folder}")
print(f"Size: {size_mb:.2f} MB")
print(f"Size: {size_gb:.2f} GB")

Folder: data
Size: 2096.86 MB
Size: 2.05 GB
